<a href="https://colab.research.google.com/github/khushipatil05/ontology-aware-protein-function-prediction/blob/main/notebooks/01_data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("NO GPU! Go to Runtime > Change runtime type > T4 GPU, then restart and run again.")

CUDA available: True
GPU: Tesla T4


In [2]:
GITHUB_REPO = "https://github.com/khushipatil05/ontology-aware-protein-function-prediction.git"
DRIVE_ROOT  = "/content/drive/MyDrive"
REPO_NAME   = "ontology-aware-protein-function-prediction"
EMBED_LIMIT = 50000      # None = all proteins
LAMBDAS     = [0.0, 0.5] # 0.0 = baseline (no hierarchy loss), 0.5 = ontology-aware

In [3]:
from google.colab import drive
drive.mount("/content/drive")

import os, subprocess
PROJECT = f"{DRIVE_ROOT}/{REPO_NAME}"
os.makedirs(PROJECT, exist_ok=True)

if os.path.exists(f"{PROJECT}/src/config.py"):
    # code already there (cloned before, or uploaded manually) -> just update if it is a git repo
    if os.path.exists(f"{PROJECT}/.git"):
        print(subprocess.run(["git", "-C", PROJECT, "pull"], capture_output=True, text=True).stdout)
    print("Code found at", PROJECT)
else:
    # PROJECT may already contain data/processed.zip, so clone into a temp dir and copy over
    subprocess.run(["rm", "-rf", "/content/_tmp_repo"])
    r = subprocess.run(["git", "clone", GITHUB_REPO, "/content/_tmp_repo"], capture_output=True, text=True)
    print(r.stdout, r.stderr)
    subprocess.run("cp -r /content/_tmp_repo/. " + PROJECT + "/", shell=True)
    assert os.path.exists(f"{PROJECT}/src/config.py"), (
        "src/config.py not found. Push your code to GitHub first, or upload the repo folder to Drive.")

%cd {PROJECT}

Mounted at /content/drive
 Cloning into '/content/_tmp_repo'...

/content/drive/MyDrive/ontology-aware-protein-function-prediction


In [4]:
!pip install -q transformers scikit-learn scipy tqdm


In [14]:
import os, json, zipfile, gzip, shutil
proc = "data/processed"
os.makedirs(proc, exist_ok=True)

# 1. Unzip data/processed.zip
if not os.path.exists(f"{proc}/terms.tsv"):
    z = "data/processed.zip"
    assert os.path.exists(z), "Upload processed.zip to " + os.path.abspath("data") + " (see guide, Step 3)"
    with zipfile.ZipFile(z) as f:
        for m in f.namelist():
            name = m.split("processed/", 1)[1] if "processed/" in m else m
            if name and not name.endswith("/"):
                out_path = f"{proc}/{name}"
                os.makedirs(os.path.dirname(out_path) or proc, exist_ok=True)
                with open(out_path, "wb") as out:
                    out.write(f.read(m))

# 2. Decompress proteins.tsv.gz if it exists
gz_path = f"{proc}/proteins.tsv.gz"
if os.path.exists(gz_path) and not os.path.exists(f"{proc}/proteins.tsv"):
    print("Decompressing proteins.tsv.gz...")
    with gzip.open(gz_path, "rb") as f_in:
        with open(f"{proc}/proteins.tsv", "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)
    print("Decompression complete.")

# 3. Final verification of required files
for need in ["proteins.tsv", "terms.tsv", "labels.npz", "hierarchy.npy", "stats.json"]:
    assert os.path.exists(f"{proc}/{need}"), f"Missing {proc}/{need}"
print(json.dumps(json.load(open(f"{proc}/stats.json")), indent=2))

{
  "n_proteins": 466802,
  "n_terms": 4181,
  "n_edges": 29525,
  "train": 373441,
  "val": 46680,
  "test": 46681
}


In [25]:
limit = f"--limit {EMBED_LIMIT}" if EMBED_LIMIT else ""
!python src/embed.py {limit} --token_budget 16000

Loading weights: 100% 534/534 [00:00<00:00, 18073.94it/s]
[transformers] EsmModel LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
shards: 100% 10/10 [38:03<00:00, 228.39s/it]
Saved (50000, 1280)


In [16]:
!ls -lh data/processed/

total 205M
-rw------- 1 root root 462K Oct  5 15:40 hierarchy.npy
-rw------- 1 root root  13M Oct  5 15:40 labels.npz
-rw------- 1 root root  96M Oct  5 15:41 proteins.tsv
-rw------- 1 root root  96M Oct  5 15:40 proteins.tsv.gz
-rw------- 1 root root  124 Oct  5 15:40 stats.json
-rw------- 1 root root 231K Oct  5 15:40 terms.tsv


In [17]:
!file data/processed/proteins.tsv

data/processed/proteins.tsv: gzip compressed data, was "proteins.tsv", last modified: Mon Oct  5 13:10:02 2026, from Unix, original size modulo 2^32 179718569


In [18]:
!file data/processed/proteins.tsv.gz

data/processed/proteins.tsv.gz: gzip compressed data, was "proteins.tsv", last modified: Mon Oct  5 13:10:02 2026, from Unix, original size modulo 2^32 179718569


In [19]:
with open("data/processed/proteins.tsv", "rb") as f:
    print(f.read(10))

b'\x1f\x8b\x08\x08\xaa\xa1\xc3j\x00\x03'


In [20]:
with open("data/processed/proteins.tsv.gz", "rb") as f:
    print(f.read(10))

b'\x1f\x8b\x08\x08\xaa\xa1\xc3j\x00\x03'


In [22]:
!rm data/processed/proteins.tsv

rm: cannot remove 'data/processed/proteins.tsv': No such file or directory


In [23]:
!gzip -dk data/processed/proteins.tsv.gz

In [24]:
!file data/processed/proteins.tsv

data/processed/proteins.tsv: ASCII text, with very long lines (1044), with CRLF line terminators


In [26]:
for lam in LAMBDAS:
    tag = "baseline" if lam == 0 else f"mlp_lam{lam}"
    print("="*20, tag, "="*20)
    !python src/train.py --lam {lam} --tag {tag}

==================== baseline ====================
epoch 01  train 0.0559  val 0.0206  (bce 0.0206 hier 0.0000)
epoch 02  train 0.0178  val 0.0143  (bce 0.0143 hier 0.0000)
epoch 03  train 0.0131  val 0.0108  (bce 0.0108 hier 0.0000)
epoch 04  train 0.0104  val 0.0089  (bce 0.0089 hier 0.0000)
epoch 05  train 0.0088  val 0.0076  (bce 0.0076 hier 0.0000)
epoch 06  train 0.0077  val 0.0069  (bce 0.0069 hier 0.0000)
epoch 07  train 0.0070  val 0.0063  (bce 0.0063 hier 0.0000)
epoch 08  train 0.0064  val 0.0059  (bce 0.0059 hier 0.0000)
epoch 09  train 0.0060  val 0.0055  (bce 0.0055 hier 0.0000)
epoch 10  train 0.0056  val 0.0052  (bce 0.0052 hier 0.0000)
epoch 11  train 0.0053  val 0.0050  (bce 0.0050 hier 0.0000)
epoch 12  train 0.0050  val 0.0049  (bce 0.0049 hier 0.0000)
epoch 13  train 0.0048  val 0.0047  (bce 0.0047 hier 0.0000)
epoch 14  train 0.0045  val 0.0046  (bce 0.0046 hier 0.0000)
epoch 15  train 0.0044  val 0.0045  (bce 0.0045 hier 0.0000)
epoch 16  train 0.0042  val 0.0044

In [27]:
for lam in LAMBDAS:
    tag = "baseline" if lam == 0 else f"mlp_lam{lam}"
    print("="*20, tag, "="*20)
    !python src/evaluate.py --tag {tag}

==================== baseline ====================
/content/drive/MyDrive/ontology-aware-protein-function-prediction/src/model.py:38: UserWarning: index_reduce() is in beta and the API may change at any time. (Triggered internally at /pytorch/aten/src/ATen/native/TensorAdvancedIndexing.cpp:1513.)
  out.index_reduce_(1, parent_idx, probs[:, child_idx], "amax", include_self=True)
Molecular Function {
 "raw": {
  "Fmax": 0.9542,
  "threshold": 0.52,
  "precision_micro": 0.9767,
  "recall_micro": 0.9536,
  "F1_micro": 0.965,
  "AUPR_micro": 0.9864,
  "AUPR_macro": 0.898
 },
 "hierarchy_enforced": {
  "Fmax": 0.9569,
  "threshold": 0.66,
  "precision_micro": 0.9803,
  "recall_micro": 0.9525,
  "F1_micro": 0.9662,
  "AUPR_micro": 0.9873,
  "AUPR_macro": 0.8984
 },
 "violation_rate_raw@0.5": 0.0071,
 "violation_rate_enforced@0.5": 0.0
}
Biological Process {
 "raw": {
  "Fmax": 0.943,
  "threshold": 0.46,
  "precision_micro": 0.9471,
  "recall_micro": 0.9178,
  "F1_micro": 0.9322,
  "AUPR_micr

In [28]:
import json, pandas as pd
rows = []
for lam in LAMBDAS:
    tag = "baseline" if lam == 0 else f"mlp_lam{lam}"
    res = json.load(open(f"results/{tag}_test_metrics.json"))
    for ont, r in res.items():
        rows.append({"model": tag, "ontology": ont,
                     "Fmax": r["raw"]["Fmax"], "AUPR_micro": r["raw"]["AUPR_micro"],
                     "AUPR_macro": r["raw"]["AUPR_macro"],
                     "violations@0.5 (raw)": r["violation_rate_raw@0.5"],
                     "violations@0.5 (enforced)": r["violation_rate_enforced@0.5"]})
pd.DataFrame(rows)

,model,ontology,Fmax,AUPR_micro,AUPR_macro,violations@0.5 (raw),violations@0.5 (enforced)
0,baseline,Molecular Function,0.9542,0.9864,0.8980,0.0071,0.0
1,baseline,Biological Process,0.9430,0.9671,0.7298,0.0071,0.0
2,baseline,Cellular Component,0.9525,0.9642,0.6919,0.0071,0.0
3,mlp_lam0.5,Molecular Function,0.9542,0.9870,0.8958,0.0048,0.0
4,mlp_lam0.5,Biological Process,0.9436,0.9677,0.7228,0.0048,0.0
5,mlp_lam0.5,Cellular Component,0.9523,0.9630,0.6824,0.0048,0.0
